# Module 5: Defense Comparison

This staged notebook loads the FedAvg baseline artifacts, runs each configured defense under the same attack recipe, and saves comparison tables and plots.


## Stage Goal

Change only the server aggregation rule while keeping the Module 4 malicious-client path, dataset split, and FedAvg round settings fixed.


## 1. Notebook Setup

Load shared helpers and make local imports work from either the repository root or the module directory.


In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd()
MODULE_DIR = cwd if cwd.name == "5_Defensive_FL" else cwd / "5_Defensive_FL"
MODULE4_DIR = MODULE_DIR.parent / "4_Adversarial_FL"
MODULE4_SRC_DIR = MODULE4_DIR / "src"
REPO_ROOT = MODULE_DIR.parent

for path in (REPO_ROOT, MODULE4_DIR, MODULE4_SRC_DIR, MODULE_DIR):
    path_str = str(path.resolve())
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

from notebook_utils import (
    load_context,
    record_config_snapshot,
    load_required_baselines,
    run_defense_comparison,
    validate_artifacts,
    validate_result_collection,
)


## 2. Configuration and Baseline Handoff

This stage expects `fedavg_baselines.ipynb` to have already written the clean and attacked FedAvg JSON files.


In [ ]:
context = load_context("defense_comparison_config.yaml", stage_name="defense_comparison")
config_snapshot_path = record_config_snapshot(context)
run_results = load_required_baselines(context)
baseline_validation = validate_result_collection(
    context,
    run_results,
    required_runs=["clean_fedavg", "attacked_fedavg"],
)
baseline_validation


## 3. Fixed-Attack Defense Comparison

Run every defense listed in `experiments.defenses`. Infeasible settings are saved as skipped rows instead of stopping the notebook.


In [ ]:
run_results, comparison_rows = run_defense_comparison(
    context,
    run_results=run_results,
)
comparison_rows


## 4. Validation and Handoff

The comparison artifacts are the inputs for result review and the optional sweep notebook.


In [ ]:
comparison_validation = validate_result_collection(context, run_results)
comparison_artifacts = validate_artifacts([
    config_snapshot_path,
    context.artifact_path("module5_defense_comparison.json"),
    context.artifact_path("module5_defense_comparison.csv"),
    context.artifact_path("module5_accuracy_curves.png"),
    context.artifact_path("module5_surrogate_poison_success_curves.png"),
    context.artifact_path("module5_global_target_label_asr_curves.png"),
    context.artifact_path("module5_accuracy_vs_defense.png"),
    context.artifact_path("module5_surrogate_poison_success_vs_defense.png"),
    context.artifact_path("module5_global_target_label_asr_vs_defense.png"),
])
{
    "validated_runs": comparison_validation,
    "artifacts": comparison_artifacts,
}
